# 212 — Raw ERSP clustering (band-aware downsampled 15×30)

Same canonical sample set as 210, but each ERSP is compressed from full-res
(129 × 300 = 38,700 features) to a neuroscience-band-aware 15 × 30 = 450 features
before clustering. Dim reduction before clustering typically lifts silhouettes 2–4×
vs raw 38k-dim space (Becht et al. 2019), and band-aware (vs uniform) spacing keeps
each EEG band (delta, theta, alpha, beta, gamma, HG, HFO sub-bands) represented by
at least one bin.

Outputs land in `outputs/clustering/{kmeans,hierarchical}/rawds/runs/<timestamp>/`.
MOBA picks up the new `rawds` feature_set automatically via index.json.


In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

from sklearn.preprocessing import StandardScaler
from functions.lf_features  import (
    downsample_ersp_to_bands,
    build_X_3d_downsampled,
    FREQ_BANDS_15_TO_400HZ,
)
from functions import lf_cluster_run as R
from functions import lf_blob_clustering_config as cfg

SCRIPT_NAME = '212_raw_downsampled_clustering.ipynb'


## Config

In [ ]:
# ── data input ─────────────────────────────
INPUT_DIR = Path(r'\\\\nasac-m2.unige.ch\\m-HumanNeuronLab\\ANALYSIS\\FLM\\Analysis_LoraFanda\\01_FBM_Analysis\\outputs\\04_ersp_LM_RAWONLY')
if not INPUT_DIR.exists():
    INPUT_DIR = Path('../01_FBM_Analysis/outputs/04_ersp_LM_RAWONLY').resolve()

# ── downsampling ───────────────────────────
# Each tuple = one output freq band (lo_hz, hi_hz). Default = 15 bands spanning
# 1..400 Hz, finer at low freq, coarser at HFO. Override here to tweak.
FREQ_BAND_EDGES = FREQ_BANDS_15_TO_400HZ
FMAX_HZ         = 500.0     # upper edge of the ORIGINAL ERSP freq axis (Hz).
                            # If your ERSP was computed with fmax=400, set this to 400.
TIME_BINS_OUT   = 30        # target time-axis length (300 -> 30 = 10x downsample)

# ── clustering ─────────────────────────────
KMEANS_K_RANGE = [10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
HC_METHOD      = 'ward'
HC_METRIC      = 'euclidean'
RANDOM_STATE   = cfg.RANDOM_STATE

print('FREQ_BAND_EDGES:')
for i, (lo, hi) in enumerate(FREQ_BAND_EDGES):
    print(f'  bin {i:2d}: {lo:>5.0f}–{hi:>5.0f} Hz')
print(f'FMAX_HZ = {FMAX_HZ}  TIME_BINS_OUT = {TIME_BINS_OUT}')
print(f'Output feature dim per sample = {len(FREQ_BAND_EDGES) * TIME_BINS_OUT}')


## Load canonical dataset (shared across 210/230/231/232/212)

In [ ]:
# Loads ERSPs from INPUT_DIR, drops non-neural channels, applies the
# high-activity gate. Same canonical sample set as 210/230/231/232 so
# cross-comparison is valid.
from functions.lf_dataset import prepare_dataset, DEFAULT_CACHE_DIR

df_meta, ersp_list, X_3d_full = prepare_dataset(INPUT_DIR, cache_dir=DEFAULT_CACHE_DIR)
print(f'\nCanonical dataset: {len(df_meta)} samples · X_3d_full.shape={X_3d_full.shape}')
df_meta.head()


## Downsample: 129 × 300 → 15 × 30

Per-band mean across the freq axis (drops everything above `FMAX_HZ` and
compresses to one bin per neuroscience band), then anti-aliased resize
of the time axis from 300 → `TIME_BINS_OUT`. Same skimage `resize` used
by `lf_minus101.downsample_minus101_map`.


In [ ]:
X_3d_ds = build_X_3d_downsampled(
    ersp_list,
    freq_band_edges=FREQ_BAND_EDGES,
    fmax_hz=FMAX_HZ,
    time_bins_out=TIME_BINS_OUT,
)
print('X_3d_ds.shape:', X_3d_ds.shape)

X_raw = X_3d_ds.reshape(X_3d_ds.shape[0], -1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
print('X_scaled.shape:', X_scaled.shape, '  (n_samples × 450)')
print('Mean ~0, std ~1?  mean[:5]:', X_scaled[:, :5].mean(axis=0).round(3))
print('                  std[:5]: ', X_scaled[:, :5].std(axis=0).round(3))


## QC: sanity-check a grid of downsampled ERSPs

Pick 20 random samples and plot their downsampled ERSP. The gross spectro-
temporal pattern should be visible — HG should be a clear stripe, beta
should be visible at low freq, baseline should be ~0.


In [ ]:
rng = np.random.default_rng(42)
qc_idx = rng.choice(len(X_3d_ds), size=min(20, len(X_3d_ds)), replace=False)

n_cols = 5
n_rows = int(np.ceil(len(qc_idx) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.6 * n_cols, 2.0 * n_rows))
axes = np.atleast_1d(axes).ravel()
vlim = float(np.percentile(np.abs(X_3d_ds[qc_idx]), 98))
for ax, i in zip(axes, qc_idx):
    ax.imshow(X_3d_ds[i], aspect='auto', origin='lower', cmap='bwr',
              vmin=-vlim, vmax=vlim, interpolation='nearest')
    m = df_meta.iloc[int(i)]
    ax.set_title(f"{m['patient_id']} {m['electrode']} {m['condition']}", fontsize=7)
    ax.set_xticks([]); ax.set_yticks([])
for ax in axes[len(qc_idx):]:
    ax.axis('off')
fig.suptitle(f'Downsampled ERSPs ({len(FREQ_BAND_EDGES)} bands × {TIME_BINS_OUT} time bins, vlim=±{vlim:.1f} dB)',
             fontsize=10)
plt.tight_layout()
plt.show()


# Clustering

Two methods on the same `X_scaled` (450 features per sample): KMeans K-sweep + Hierarchical K-sweep.
Each `fit_and_save` call writes a self-contained run dir at `outputs/clustering/{kmeans,hierarchical}/rawds/runs/<timestamp>/`.


In [ ]:
manifest_km = R.fit_and_save(
    X_scaled,
    df_keep=df_meta,
    method='kmeans',
    feature_set='rawds',
    params={'k_range': KMEANS_K_RANGE, 'random_state': RANDOM_STATE, 'n_init': 20},
    scaler=scaler,
    method_label='K-Means',
    feature_set_label='Raw ERSP (band-aware downsampled 15×30)',
    notebook=SCRIPT_NAME,
)
BEST_K = manifest_km['summary']['best_k']
print(f'Best K (KMeans/rawds, by silhouette): {BEST_K}')


In [ ]:
manifest_hc = R.fit_and_save(
    X_scaled,
    df_keep=df_meta,
    method='hierarchical',
    feature_set='rawds',
    params={'linkage': HC_METHOD, 'metric': HC_METRIC, 'k_range': KMEANS_K_RANGE},
    scaler=scaler,
    method_label='Hierarchical (Ward)',
    feature_set_label='Raw ERSP (band-aware downsampled 15×30)',
    notebook=SCRIPT_NAME,
)
print(f'Best K (HC/rawds, by silhouette): {manifest_hc["summary"]["best_k"]}')


## Per-cluster centroid PNGs (for the MOBA cluster chips)

Mean downsampled ERSP per cluster, plotted as the 15×30 imshow. Cheap to re-run.


In [ ]:
# BACKFILL_CENTROIDS — per-cluster mean of the 15×30 downsampled ERSP
import json

CLUSTERING_DIR = Path(R.DEFAULT_OUTPUTS_ROOT)
INDEX_PATH = CLUSTERING_DIR / 'index.json'

def _save_per_cluster_centroid_pngs_rawds(manifest, X_3d_local, *, vlim=5.0):
    if manifest['feature_set'] != 'rawds':
        return 0
    run_dir = CLUSTERING_DIR / manifest['method'] / manifest['feature_set'] / 'runs' / manifest['run_id']
    cluster_col = f"cluster_{manifest['method']}_{manifest['feature_set']}"
    df = pd.read_csv(run_dir / 'labels.csv')
    if cluster_col not in df.columns:
        cands = [c for c in df.columns if c.startswith('cluster_')]
        if not cands: return 0
        cluster_col = cands[0]
    labels = df[cluster_col].to_numpy()
    if len(labels) != X_3d_local.shape[0]:
        print(f"  [skip] {manifest['run_id']}: labels ({len(labels)}) vs X_3d_ds ({X_3d_local.shape[0]}) mismatch")
        return 0
    out_dir = run_dir / 'cluster_centroids'
    out_dir.mkdir(parents=True, exist_ok=True)
    uniq = sorted(int(c) for c in np.unique(labels))
    for c in uniq:
        idx = np.where(labels == c)[0]
        mean_ersp = X_3d_local[idx].mean(axis=0)   # already (n_freq_out, n_time_out)
        fig, ax = plt.subplots(figsize=(2, 1.5))
        ax.imshow(mean_ersp, aspect='auto', origin='lower',
                  cmap='bwr', vmin=-vlim, vmax=vlim, interpolation='nearest')
        ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values(): s.set_visible(False)
        fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
        fig.savefig(out_dir / f'cluster_{int(c):02d}.png', dpi=80, bbox_inches='tight', pad_inches=0)
        plt.close(fig)
    return len(uniq)

if INDEX_PATH.exists():
    with open(INDEX_PATH) as f:
        idx = json.load(f)
    runs = [r for r in idx.get('runs', []) if r['feature_set'] == 'rawds']
    print(f'Backfilling for {len(runs)} rawds runs...')
    for run in runs:
        mp = CLUSTERING_DIR / run['path'] / 'manifest.json'
        if not mp.exists(): continue
        manifest = json.loads(mp.read_text())
        n = _save_per_cluster_centroid_pngs_rawds(manifest, X_3d_ds)
        if n: print(f"  [{manifest['method']}/rawds] {manifest['run_id']} -> {n} PNGs")
    print('Done.')
